# **1. Chuẩn bị dữ liệu**

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
def load_image_from_folder(folder, img_size=(128, 128)):
    images = []
    labels = []
    
    # Lấy và sắp xếp danh sách các lớp: ['Green', 'Overripe', 'Ripe', 'Semi-ripe']
    class_names = sorted([d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))])

    for label_idx, class_name in enumerate(class_names):
        class_folder = os.path.join(folder, class_name)
        valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
        
        for filename in os.listdir(class_folder):
            if filename.lower().endswith(valid_extensions):
                img_path = os.path.join(class_folder, filename)
                img = cv2.imread(img_path)
                
                if img is not None:
                    if img_size is not None:
                        img = cv2.resize(img, img_size)
                    images.append(img)
                    labels.append(label_idx)

    return images, labels, class_names

In [3]:
dataset_root = "/kaggle/input/datasets/thitrmnguyn/banana/Augmented Banana Ripeness Detection Dataset"

images, labels, class_names  = load_image_from_folder(dataset_root)

print(f"Tổng số ảnh nạp được: {len(images)}")
print(f"Phân bố nhãn: {np.bincount(labels)}")

Tổng số ảnh nạp được: 2457
Phân bố nhãn: [636 609 603 609]


In [4]:
x_train, x_test, y_train, y_test = train_test_split(
    images, 
    labels, 
    test_size = 0.2, 
    random_state = 42, 
    stratify=labels
)
print(f"Số lượng ảnh tập train: {len(x_train)}")
print(f"Số lượng ảnh tập test : {len(x_test)}")
print(f"Phân bố nhãn tập train: {np.bincount(y_train)}")
print(f"Phân bố nhãn tập test : {np.bincount(y_test)}")

Số lượng ảnh tập train: 1965
Số lượng ảnh tập test : 492
Phân bố nhãn tập train: [509 487 482 487]
Phân bố nhãn tập test : [127 122 121 122]


# **2. Rút trích đặc trưng** 

In [5]:
import cv2
import numpy as np
from skimage.feature import local_binary_pattern, hog

# 1. Đặc trưng HSV : bỏ đi đặc trưng V
def extract_hsv(image, h_bins=16, s_bins=8):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [h_bins, s_bins], [0, 180, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    return hist

# 2. Đặc trưng LBP (Local Binary Patterns)
def extract_lbp(image, P=8, R=1):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, P=P, R=R, method="uniform")
    n_bins = int(lbp.max() + 1)
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins), density=True)
    return hist

# 3. Đặc trưng HOG (Histogram of Oriented Gradients)
def extract_hog(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_resized = cv2.resize(gray, (64, 64))
    hog_feat = hog(
        gray_resized,
        orientations=8,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )
    return hog_feat

In [6]:
# 1. Rút trích HSV
X_train_hsv = np.array([extract_hsv(img) for img in x_train])
X_test_hsv = np.array([extract_hsv(img) for img in x_test])

# 2. Rút trích LBP
X_train_lbp = np.array([extract_lbp(img) for img in x_train])
X_test_lbp = np.array([extract_lbp(img) for img in x_test])

# 3. Kết hợp HSV + LBP
X_train_hsv_lbp = np.hstack([X_train_hsv, X_train_lbp])
X_test_hsv_lbp = np.hstack([X_test_hsv, X_test_lbp])

# 4. Rút trích HOG và kết hợp HSV + HOG
X_train_hog = np.array([extract_hog(img) for img in x_train])
X_test_hog = np.array([extract_hog(img) for img in x_test])

X_train_hsv_hog = np.hstack([X_train_hsv, X_train_hog])
X_test_hsv_hog = np.hstack([X_test_hsv, X_test_hog])

print("Kích thước các bộ đặc trưng trên tập Train:")
print(f"- SVM 1 (HSV)      : {X_train_hsv.shape}")
print(f"- SVM 2 (LBP)      : {X_train_lbp.shape}")
print(f"- SVM 3 (HSV + LBP): {X_train_hsv_lbp.shape}")
print(f"- SVM 4 (HSV + HOG): {X_train_hsv_hog.shape}")

Kích thước các bộ đặc trưng trên tập Train:
- SVM 1 (HSV)      : (1965, 128)
- SVM 2 (LBP)      : (1965, 10)
- SVM 3 (HSV + LBP): (1965, 138)
- SVM 4 (HSV + HOG): (1965, 416)


# **3. Huấn luyện SVM với GridSearchCV**

In [7]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Lưới tham số C và gamma theo yêu cầu mục 5
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'kernel': ['rbf']
}

experiments = {
    "SVM 1 (HSV)": (X_train_hsv, X_test_hsv),
    "SVM 2 (LBP)": (X_train_lbp, X_test_lbp),
    "SVM 3 (HSV + LBP)": (X_train_hsv_lbp, X_test_hsv_lbp),
    "SVM 4 (HSV + HOG)": (X_train_hsv_hog, X_test_hsv_hog)
}

trained_models = {}

for name, (X_tr, X_te) in experiments.items():
    print(f"Đang huấn luyện {name}...")
    grid = GridSearchCV(SVC(random_state=42), param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
    grid.fit(X_tr, y_train)
    trained_models[name] = grid
    print(f"-> Hoàn tất: Best Params = {grid.best_params_} | Best CV F1 = {grid.best_score_:.4f}")

Đang huấn luyện SVM 1 (HSV)...
-> Hoàn tất: Best Params = {'C': 100, 'gamma': 'scale', 'kernel': 'rbf'} | Best CV F1 = 0.9344
Đang huấn luyện SVM 2 (LBP)...
-> Hoàn tất: Best Params = {'C': 100, 'gamma': 'scale', 'kernel': 'rbf'} | Best CV F1 = 0.7132
Đang huấn luyện SVM 3 (HSV + LBP)...
-> Hoàn tất: Best Params = {'C': 100, 'gamma': 1, 'kernel': 'rbf'} | Best CV F1 = 0.9404
Đang huấn luyện SVM 4 (HSV + HOG)...
-> Hoàn tất: Best Params = {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'} | Best CV F1 = 0.8533


In [9]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

summary_results = []

for name, (X_tr, X_te) in experiments.items():
    grid = trained_models[name]
    y_pred = grid.predict(X_te)
    
    summary_results.append({
        "Thí nghiệm": name,
        "Số chiều": X_tr.shape[1],
        "Best C": grid.best_params_['C'],
        "Best Gamma": grid.best_params_['gamma'],
        "CV F1-Score": round(grid.best_score_, 4),
        "Test Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Test F1-Macro": round(f1_score(y_test, y_pred, average='macro'), 4)
    })

df_results = pd.DataFrame(summary_results)
print("\n=== BẢNG SO SÁNH CÁC BỘ ĐẶC TRƯNG TRÊN SVM ===")
display(df_results)

# Tìm mô hình tốt nhất
best_exp_name = df_results.sort_values(by="Test F1-Macro", ascending=False).iloc[0]["Thí nghiệm"]
print(f"\n=> Bộ đặc trưng cho kết quả tốt nhất: {best_exp_name}")

best_model = trained_models[best_exp_name]
X_best_test = experiments[best_exp_name][1]
y_best_pred = best_model.predict(X_best_test)

print(f"\n=== CLASSIFICATION REPORT CỦA {best_exp_name} TRÊN TEST ===")
print(classification_report(y_test, y_best_pred, target_names=class_names))


=== BẢNG SO SÁNH CÁC BỘ ĐẶC TRƯNG TRÊN SVM ===


,Thí nghiệm,Số chiều,Best C,Best Gamma,CV F1-Score,Test Accuracy,Test F1-Macro
0,SVM 1 (HSV),128,100,scale,0.9344,0.9715,0.9714
1,SVM 2 (LBP),10,100,scale,0.7132,0.7256,0.7241
2,SVM 3 (HSV + LBP),138,100,1,0.9404,0.9756,0.9754
3,SVM 4 (HSV + HOG),416,100,0.01,0.8533,0.8882,0.8870



=> Bộ đặc trưng cho kết quả tốt nhất: SVM 3 (HSV + LBP)

=== CLASSIFICATION REPORT CỦA SVM 3 (HSV + LBP) TRÊN TEST ===
              precision    recall  f1-score   support

       Green       1.00      0.99      1.00       127
    Overripe       0.96      0.99      0.98       122
        Ripe       0.98      0.97      0.97       121
   Semi-ripe       0.96      0.95      0.95       122

    accuracy                           0.98       492
   macro avg       0.98      0.98      0.98       492
weighted avg       0.98      0.98      0.98       492

